In [ ]:
from pathlib import Path
import numpy as np
import textgrid  # pip install textgrid


def read_mfa_word_intervals(tg_path):
    """MFA TextGrid에서 (start_sec, end_sec, word) 리스트."""
    tg = textgrid.TextGrid.fromFile(str(tg_path))
    tier = next(t for t in tg.tiers if t.name.lower() == "words")
    return [
        (iv.minTime, iv.maxTime, iv.mark.strip().lower())
        for iv in tier
        if iv.mark.strip()
    ]


def read_ref(wrd_path, sr=16000):
    ref = []
    for line in open(wrd_path):
        line = line.strip()
        if not line:
            continue
        s, e, w = line.split(maxsplit=2)
        ref.append((int(s) / sr, int(e) / sr, w.strip().lower()))
    return ref


def lcs_match(ref_words, hyp_words):
    n, m = len(ref_words), len(hyp_words)
    dp = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(n):
        for j in range(m):
            if ref_words[i] == hyp_words[j]:
                dp[i + 1][j + 1] = dp[i][j] + 1
            else:
                dp[i + 1][j + 1] = max(dp[i][j + 1], dp[i + 1][j])
    pairs = []
    i, j = n, m
    while i > 0 and j > 0:
        if ref_words[i - 1] == hyp_words[j - 1]:
            pairs.append((i - 1, j - 1))
            i -= 1
            j -= 1
        elif dp[i - 1][j] >= dp[i][j - 1]:
            i -= 1
        else:
            j -= 1
    pairs.reverse()
    return pairs


def report(name, errs):
    e = np.array(errs)
    print(f"{name:12s}  n={len(e):7d}  WBE={e.mean()*1000:6.2f} ms  "
          + f"P10={100*(e<=.010).mean():5.2f}  P25={100*(e<=.025).mean():5.2f}  "
          + f"P50={100*(e<=.050).mean():5.2f}  P100={100*(e<=.100).mean():5.2f}")

# TIMIT

In [2]:
TG_ROOT  = Path("/shared/data_zfs/blue2959/timit_test_aligned")
WRD_ROOT = Path("/shared/data_zfs/blue2959/TIMIT/TEST")

wrd_index = {}
for p in WRD_ROOT.rglob("*.WRD"):
    wrd_index[(p.parent.name, p.stem)] = p

tg_paths = sorted(TG_ROOT.rglob("*.TextGrid"))
print(f"{len(tg_paths)} TextGrids, {len(wrd_index)} WRD files")

all_errs, all_starts, all_ends = [], [], []
n_missing = 0
total_bounds = kept_bounds = 0

for tg in tg_paths:
    spk = tg.parent.name
    utt = tg.stem[len(spk) + 1:]          # FADG0_SA1 -> SA1

    wrd = wrd_index.get((spk, utt))
    if wrd is None:
        n_missing += 1
        continue

    ref = read_ref(wrd)
    total_bounds += 2 * len(ref)

    hyp = read_mfa_word_intervals(tg)
    pairs = lcs_match([w for *_, w in ref], [w for *_, w in hyp])

    for ri, hi in pairs:
        rs, re_, _ = ref[ri]
        hs, he, _ = hyp[hi]
        all_starts.append(abs(hs - rs))
        all_ends.append(abs(he - re_))
        all_errs.extend([abs(hs - rs), abs(he - re_)])
        kept_bounds += 2

print(f"\nmissing={n_missing}")
print(f"boundary coverage = {100*kept_bounds/total_bounds:.2f}%  "
      + f"({kept_bounds}/{total_bounds})\n")
report("start+end", all_errs)
report("start only", all_starts)
report("end only", all_ends)

1680 TextGrids, 1680 WRD files

missing=0
boundary coverage = 99.97%  (29098/29106)

start+end     n=  29098  WBE= 18.66 ms  P10=46.31  P25=76.59  P50=92.37  P100=98.48
start only    n=  14549  WBE= 17.69 ms  P10=46.21  P25=77.46  P50=94.01  P100=98.84
end only      n=  14549  WBE= 19.62 ms  P10=46.41  P25=75.72  P50=90.73  P100=98.12


# Buckeye

In [3]:
import re, json
from pathlib import Path
from collections import Counter
import soundfile as sf
from praatio import textgrid as tgio

from nd_aligner.benchmark.buckeye.buckeye_to_timit_eval_format import (
    entry_values,
    is_special_token,
    find_tier_name,
    seconds_to_sample,
    select_words_for_utterance,
)

BENCH = Path("/shared/data_zfs/blue2959/Buckeye-benchmark")
MIN_WORDS = 4

offsets = {}   # (spk, "s0101a_chunk_0000") -> crop_start_sec

for tg_path in sorted(BENCH.glob("s*/s*.TextGrid")):
    spk = tg_path.parent.name
    rec = tg_path.stem
    wav = tg_path.with_suffix(".wav")
    if not wav.exists():
        continue

    info = sf.info(wav)
    sr = info.samplerate
    total = int(info.frames)

    tg = tgio.openTextgrid(tg_path, includeEmptyIntervals=False)
    names = list(tg.tierNames)
    utt_tier = tg.getTier(find_tier_name(names, spk, "utterance")).entries
    wrd_tier = tg.getTier(find_tier_name(names, spk, "words")).entries

    idx = 0
    for entry in utt_tier:
        us, ue, ul = entry_values(entry)
        ul = ul.strip()
        if not ul:
            continue
        words = select_words_for_utterance(wrd_tier, us, ue)
        if not words:
            continue
        if any(is_special_token(l) for _, _, l in words) or \
           any(is_special_token(t) for t in ul.split()):
            continue
        if len(words) < MIN_WORDS:
            continue
        cs = max(0, min(seconds_to_sample(us, sr), total))
        ce = max(0, min(seconds_to_sample(ue, sr), total))
        if ce <= cs:
            continue

        offsets[(spk, f"{rec}_chunk_{idx:04d}")] = cs / sr
        idx += 1

print(len(offsets), "offsets")   # 19273 이어야 함
json.dump({f"{k[0]}/{k[1]}": v for k, v in offsets.items()},
          open("buckeye_offsets.json", "w"))

19273 offsets


In [ ]:
import json
from tqdm import tqdm
from collections import defaultdict

offsets = {tuple(k.split("/")): v for k, v in
           json.load(open("buckeye_offsets.json")).items()}

TG_ROOT = Path("/shared/data_zfs/blue2959/Buckeye-aligned")
GRID_ROOT = Path("/shared/data_zfs/blue2959/Buckeye-grid")

hyp_cache = {}

all_errs, all_starts, all_ends = [], [], []
total_bounds = kept_bounds = 0
n_missing = 0

for wrd in tqdm(sorted(GRID_ROOT.rglob("*.WRD"))):
    spk = wrd.parent.name
    key = (spk, wrd.stem)
    if key not in offsets:
        n_missing += 1
        continue
    off = offsets[key]

    rec = wrd.stem.split("_chunk_")[0]
    if (spk, rec) not in hyp_cache:
        tg = TG_ROOT / spk / f"{rec}.TextGrid"
        if not tg.exists():
            n_missing += 1
            continue
        hyp_cache[(spk, rec)] = read_mfa_word_intervals(tg)
    hyp_all = hyp_cache[(spk, rec)]

    ref = read_ref(wrd)
    ref = [(s + off, e + off, w) for s, e, w in ref]
    total_bounds += 2 * len(ref)

    # chunk 시간 창으로 hyp 범위 축소
    lo, hi_t = ref[0][0] - 0.5, ref[-1][1] + 0.5
    hyp = [x for x in hyp_all if x[1] > lo and x[0] < hi_t]

    pairs = lcs_match([w for *_, w in ref], [w for *_, w in hyp])
    for ri, hi in pairs:
        rs, re_, _ = ref[ri]
        hs, he, _ = hyp[hi]
        all_starts.append(abs(hs - rs))
        all_ends.append(abs(he - re_))
        all_errs.extend([abs(hs - rs), abs(he - re_)])
        kept_bounds += 2

print(f"\nmissing={n_missing}")
print(f"boundary coverage = {100*kept_bounds/total_bounds:.2f}%  "
      + f"({kept_bounds}/{total_bounds})\n")
report("start+end", all_errs)
report("start only", all_starts)
report("end only", all_ends)

100%|██████████| 19273/19273 [00:12<00:00, 1583.68it/s]



missing=0
boundary coverage = 99.89%  (439372/439858)

start+end     n= 439372  WBE= 21.49 ms  P10=46.94  P25=76.68  P50=91.68  P100=97.36
start only    n= 219686  WBE= 21.32 ms  P10=47.26  P25=77.00  P50=91.82  P100=97.46
end only      n= 219686  WBE= 21.65 ms  P10=46.63  P25=76.35  P50=91.54  P100=97.26
